# CNN-EXP-046-B — Recovery V3: CPU head + restart-safe train-contig CV

This notebook **continues the existing run_001**. It does not recreate Stage 1 or Stage 2 and does not touch validation.

Accepted state before V3:
- Stage 1 manifest/population/LOW_TP is sealed.
- Stage 2 feature cache is complete: **173,887 / 173,887** rows.
- Fold 1 A/B training completed on DirectML, but the kernel restarted before any held-out-contig CV AP was produced. Those legacy models are preserved but excluded.

V3 makes two runtime-only recovery changes before any CV AP is seen: the tiny regional heads train/infer on CPU, while WIDTH512 remains frozen on DirectML; and held-out-contig inference is written as atomic **16-tile shards**, so a restart loses at most the current shard.

The notebook stops after train-contig CV selection. **Validation is intentionally not instantiated here.**


In [1]:
from pathlib import Path
import gc
import hashlib
import inspect
import json
import math
import os
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml
from IPython.display import display
from sklearn.metrics import roc_auc_score

# ============================================================
# HARD ENVIRONMENT / PROJECT GUARDS
# ============================================================
EXPECTED_PYTHON = os.path.normcase(os.path.abspath(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
))
ACTUAL_PYTHON = os.path.normcase(os.path.abspath(sys.executable))
assert ACTUAL_PYTHON == EXPECTED_PYTHON, (
    f"Wrong interpreter: {sys.executable}\nExpected: {EXPECTED_PYTHON}"
)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src" / "ml_project").is_dir()
)
EXPECTED_PROJECT_ROOT = Path(r"D:\Projects\EPIC\EPIC").resolve()
assert os.path.normcase(str(PROJECT_ROOT)) == os.path.normcase(str(EXPECTED_PROJECT_ROOT)), PROJECT_ROOT

AGENTS_PATH = PROJECT_ROOT.parent / "AGENTS.md"
assert AGENTS_PATH.is_file(), f"Missing required {AGENTS_PATH}"
AGENTS_SHA256 = hashlib.sha256(AGENTS_PATH.read_bytes()).hexdigest()
EXPECTED_AGENTS_SHA256 = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
assert AGENTS_SHA256 == EXPECTED_AGENTS_SHA256, (
    "AGENTS.md changed since the accepted EXP046 recovery. Review it before continuing.\n"
    f"Current: {AGENTS_SHA256}\nExpected: {EXPECTED_AGENTS_SHA256}"
)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

# ============================================================
# FROZEN SCIENTIFIC CONSTANTS
# ============================================================
EXPERIMENT_ID = "CNN-EXP-046-B"
REGION_SIZE = 32
GRID_ORIGIN = 0
REGIONS_PER_TILE = 32
NEIGHBOR_REGIONS = 2
CONTEXT_REGIONS = 5
SCORE_STAT_CHANNELS = 4
BACKBONE_POOL_CHANNELS = 1024
VARIANT_CHANNELS = {"A": SCORE_STAT_CHANNELS, "B": SCORE_STAT_CHANNELS + BACKBONE_POOL_CHANNELS}

ALPHAS = (0.0, 0.25, 0.5, 1.0, 2.0)
NEG_PER_STRATUM_PER_CONTIG = 2_500
NEGATIVE_STRATA = ("0_1", "1_5", "5_15", "15_100")
SAMPLE_SEED = 460_600
HEAD_SEED = 460_601
HEAD_EPOCHS = 4
HEAD_BATCH_SIZE = 256
HEAD_PEAK_LR = 1e-3
HEAD_FINAL_LR = 1e-4
HEAD_WARMUP_FRACTION = 0.05
HEAD_WEIGHT_DECAY = 1e-4
EVAL_INFERENCE_BATCH_SIZE = 1
CV_SHARD_TILES = 16
STAT_CHUNK = 128

TRAINING_SCHEDULE = {
    "type": "optimizer_step_warmup_plus_cosine",
    "epochs": HEAD_EPOCHS,
    "warmup_fraction": HEAD_WARMUP_FRACTION,
    "peak_lr": HEAD_PEAK_LR,
    "final_lr": HEAD_FINAL_LR,
}

BUDGET_MULTIPLIERS = (1, 2, 5, 10)
REGIONAL_RECALL_DIAGNOSTIC_TARGETS = (0.99, 0.98, 0.95)
SUBMISSION_LEVELS = 100_001

EXPECTED_TRAIN_VALID_REGIONS = 7_284_784
EXPECTED_TRAIN_ACTIVE_REGIONS = 83_887
EXPECTED_TRAIN_EMPTY_REGIONS = 7_200_897
EXPECTED_TRAIN_SAMPLE_ROWS = 173_887
EXPECTED_TRAIN_SAMPLE_ACTIVE = 83_887
EXPECTED_TRAIN_SAMPLE_EMPTY = 90_000
EXPECTED_TRAIN_LOWTP = 42_613

EXPECTED_BASELINE_AP5 = 0.09763009437871803
EXPECTED_ORACLE32_AP5 = 0.23831825797792802

EXP045_RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029" / "run_001"
)
CHECKPOINT_PATH = EXP045_RUN_DIR / "width512_50k_final.pt"
assert CHECKPOINT_PATH.is_file(), CHECKPOINT_PATH

EXP046_RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP046_REGIONAL_ACTIVITY_GATE" / "run_001"
)
FRONTIER_CSV = EXP046_RUN_DIR / "feasibility_frontier_r32" / "r32_target_ap_crossings.csv"
ORACLE_JSON = EXP046_RUN_DIR / "oracle_regional_gate_summary.json"
assert FRONTIER_CSV.is_file() and ORACLE_JSON.is_file()
frontier_crossings = pd.read_csv(FRONTIER_CSV)
frontier_099 = frontier_crossings[np.isclose(frontier_crossings["active_region_recall_target"], 0.99)].iloc[0]
assert abs(float(frontier_099["median_first_empty_suppression_fraction"]) - 0.10) < 1e-5
assert abs(float(frontier_099["median_ap1"]) - 0.20308937905110025) < 1e-12
accepted_oracle = json.loads(ORACLE_JSON.read_text(encoding="utf-8"))
assert int(accepted_oracle["primary_region_size"]) == REGION_SIZE
assert abs(float(accepted_oracle["primary_oracle_ap1"]) - EXPECTED_ORACLE32_AP5) <= 1e-12

RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32" / "run_001"
)
assert RUN_DIR.is_dir(), f"Recovery requires existing run_001: {RUN_DIR}"
CACHE_DIR = RUN_DIR / "_train_sample_cache"
FOLD_MODEL_DIR = RUN_DIR / "fold_models"
CPU_FOLD_MODEL_DIR = FOLD_MODEL_DIR / "cpu_v3"
CV_V3_DIR = RUN_DIR / "_cv_eval_v3"
assert CACHE_DIR.is_dir()
FOLD_MODEL_DIR.mkdir(exist_ok=True)
CPU_FOLD_MODEL_DIR.mkdir(exist_ok=True)
CV_V3_DIR.mkdir(exist_ok=True)

OUTPUTS = {
    "prereg": RUN_DIR / "EXP046B_preregistration.json",
    "sample_manifest": RUN_DIR / "train_region_sample_manifest.csv",
    "sampling_summary": RUN_DIR / "train_sampling_summary.csv",
    "train_lowtp": RUN_DIR / "train_fixed_lowtp_coordinates.csv",
    "channel_stats": RUN_DIR / "sample_channel_sufficient_stats.npz",
    "cv_fold_ap": RUN_DIR / "cv_fold_ap5.csv",
    "cv_pooled_ap": RUN_DIR / "cv_pooled_ap5.csv",
    "cv_region_diag": RUN_DIR / "cv_regional_diagnostics.csv",
    "cv_budget": RUN_DIR / "cv_fixed_budget_recall.csv",
    "cv_raw_audit": RUN_DIR / "cv_selected_raw_ap_by_contig.csv",
    "selected_config": RUN_DIR / "selected_config_from_train_cv.json",
    "cv_plot": RUN_DIR / "cv_alpha_curves.png",
}

# No CV selection / validation result may already exist.
for p in OUTPUTS.values():
    if p.name not in {
        "EXP046B_preregistration.json",
        "train_region_sample_manifest.csv",
        "train_sampling_summary.csv",
        "train_fixed_lowtp_coordinates.csv",
        "sample_channel_sufficient_stats.npz",
    }:
        assert not p.exists(), f"Scientific CV output already exists; stop and inspect: {p}"
for forbidden in (
    RUN_DIR / "selected_config_from_train_cv.json",
    RUN_DIR / "validation_nucleotide_metrics.csv",
    RUN_DIR / "EXP046B_RESULT.json",
):
    assert not forbidden.exists(), f"Post-CV/validation artifact exists; do not run V3 blindly: {forbidden}"

print("Project:", PROJECT_ROOT)
print("Interpreter:", sys.executable)
print("AGENTS SHA256:", AGENTS_SHA256)
print("Continuing existing run_001:", RUN_DIR)
print("V3 policy: CPU regional heads; DirectML only for frozen WIDTH512 backbone inference")


Project: D:\Projects\EPIC\EPIC
Interpreter: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
AGENTS SHA256: bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3
Continuing existing run_001: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32\run_001
V3 policy: CPU regional heads; DirectML only for frozen WIDTH512 backbone inference


In [2]:
# ============================================================
# LOAD DATA / FROZEN EXP045 / GEOMETRY SMOKE
# ============================================================
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

TRAIN_CONTIGS = sorted(set(split.intervals("train")["contig"].astype(str)))
VALIDATION_CONTIGS = sorted(set(split.intervals("validation")["contig"].astype(str)))
assert len(TRAIN_CONTIGS) == 9
assert len(VALIDATION_CONTIGS) == 3
assert set(TRAIN_CONTIGS).isdisjoint(VALIDATION_CONTIGS)
ALL_CONTIGS = TRAIN_CONTIGS + VALIDATION_CONTIGS
CONTIG_TO_CODE = {contig: i for i, contig in enumerate(ALL_CONTIGS)}

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)
assert profile_config.target_length == REGION_SIZE * REGIONS_PER_TILE
assert profile_config.context_flank >= REGION_SIZE * NEIGHBOR_REGIONS

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model = create_cnn_presence_intensity(channels=512)
model.load_state_dict(checkpoint["model_state_dict"])
assert int(checkpoint["step"]) == 50_000
assert int(model.backbone.stem.out_channels) == 512
assert int(model.profile_head.in_channels) == 512
assert int(model.profile_head.out_channels) == 2
for p in model.parameters():
    p.requires_grad_(False)

device = torch_directml.device()
model = model.to(device)
model.eval()

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
train_tiles = train_generator.tile_index.copy().reset_index(drop=True)
required = {"contig", "target_start", "target_end", "allowed_position_strand"}
missing = required - set(train_tiles.columns)
assert not missing, ("train", missing)
assert ((train_tiles["target_end"] - train_tiles["target_start"]) == profile_config.target_length).all()
assert (train_tiles["target_start"].astype(np.int64) % REGION_SIZE == 0).all()
assert (train_tiles["target_end"].astype(np.int64) % REGION_SIZE == 0).all()

# EXP046-B deliberately does not instantiate/read the validation generator here.
# Validation inference and validation labels are first touched only after the CV-selected config is frozen.

# Verify x_both_strands ordering for B>1 against the established helper.
smoke_rows = train_tiles.iloc[:2]
smoke_batch = train_generator.make_batch(smoke_rows)
with torch.no_grad():
    smoke_x = smoke_batch.x_both_strands.to(device)
    smoke_feat = model.backbone(smoke_x)
    smoke_out = model.profile_head(smoke_feat)
    smoke_plus, smoke_minus_reverse, _, _ = forward_both_strands_multitask(
        model,
        smoke_x,
        batch_size=2,
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

c = profile_config.context_flank
L = profile_config.target_length
assert smoke_out.shape[0] == 4
assert torch.allclose(smoke_out[:2, 0:1, c:c+L].cpu(), smoke_plus.cpu(), atol=1e-6, rtol=1e-6)
assert torch.allclose(smoke_out[2:, 0:1, c:c+L].cpu(), smoke_minus_reverse.cpu(), atol=1e-6, rtol=1e-6)
assert smoke_feat.shape[1] == 512
assert smoke_feat.shape[-1] >= L + 2 * c

del smoke_x, smoke_feat, smoke_out, smoke_plus, smoke_minus_reverse, smoke_batch
gc.collect()

print("Frozen EXP045 geometry/order smoke: PASS")
print("Train tiles:", len(train_tiles))
print("Train positives:", int(train_generator.total_positive))
print("Validation generator not instantiated before CV selection: PASS")


Frozen EXP045 geometry/order smoke: PASS
Train tiles: 130525
Train positives: 170443
Validation generator not instantiated before CV selection: PASS


In [3]:
# ============================================================
# METRIC / REGION / FEATURE HELPERS
# ============================================================
def sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x, dtype=np.float64)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    ex = np.exp(x[~positive])
    out[~positive] = ex / (1.0 + ex)
    return out.astype(np.float32)


def probability_bin_5dp_from_logit(logit):
    probability = sigmoid_numpy(logit)
    return np.rint(np.clip(probability, 0.0, 1.0) * 100_000).astype(np.uint32)


def update_histogram(hist, bins):
    bins = np.asarray(bins, dtype=np.int64)
    if bins.size:
        hist += np.bincount(bins, minlength=SUBMISSION_LEVELS).astype(np.int64)


def ap_from_histograms(positive_hist, total_hist):
    positives = int(positive_hist.sum())
    assert positives > 0
    pos_desc = positive_hist[::-1]
    total_desc = total_hist[::-1]
    tp = np.cumsum(pos_desc, dtype=np.int64)
    predicted = np.cumsum(total_desc, dtype=np.int64)
    precision = tp / np.maximum(predicted, 1)
    return float(np.sum((pos_desc / positives) * precision))


def exact_ap_from_scores(scores, labels):
    """Average precision on raw float32 scores with score ties grouped exactly."""
    scores = np.asarray(scores, dtype=np.float32)
    labels = np.asarray(labels, dtype=bool)
    assert scores.ndim == labels.ndim == 1
    assert len(scores) == len(labels)
    n_pos = int(labels.sum())
    assert n_pos > 0
    order = np.argsort(scores, kind="quicksort")[::-1]
    s = scores[order]
    y = labels[order]
    cumulative_tp = np.cumsum(y, dtype=np.int32)
    # End index of each distinct-score group. AP uses precision after the whole tie group.
    group_end = np.flatnonzero(np.r_[s[1:] != s[:-1], True])
    tp_end = cumulative_tp[group_end].astype(np.float64)
    tp_before = np.r_[0.0, tp_end[:-1]]
    positives_in_group = tp_end - tp_before
    precision_end = tp_end / (group_end.astype(np.float64) + 1.0)
    ap = float(np.sum((positives_in_group / n_pos) * precision_end))
    del order, s, y, cumulative_tp, group_end, tp_end, tp_before, positives_in_group, precision_end
    gc.collect()
    return ap


BIT_SHIFTS = np.arange(REGION_SIZE, dtype=np.uint32)

def bool_rows_to_u32(mask_2d):
    mask_2d = np.asarray(mask_2d, dtype=bool)
    assert mask_2d.ndim == 2 and mask_2d.shape[1] == REGION_SIZE
    weights = (np.uint32(1) << BIT_SHIFTS)[None, :]
    return np.sum(mask_2d.astype(np.uint32) * weights, axis=1, dtype=np.uint32)


def u32_to_bool_rows(values):
    values = np.asarray(values, dtype=np.uint32)
    return ((values[:, None] >> BIT_SHIFTS[None, :]) & np.uint32(1)).astype(bool)


KEY_SHIFT = np.uint64(56)
REGION_MASK = (np.uint64(1) << KEY_SHIFT) - np.uint64(1)

def pack_region_keys(contig_code, strand_code, region_ids):
    region_ids = np.asarray(region_ids, dtype=np.uint64)
    assert np.all(region_ids <= REGION_MASK)
    prefix = np.uint64(int(contig_code) * 2 + int(strand_code))
    return (prefix << KEY_SHIFT) | region_ids


def unpack_region_key(key):
    key = np.uint64(key)
    prefix = int(key >> KEY_SHIFT)
    return prefix // 2, prefix % 2, int(key & REGION_MASK)


def rank_strata_from_risk(keys, risks):
    """Return negative-stratum label per row, deterministic under score ties."""
    keys = np.asarray(keys, dtype=np.uint64)
    risks = np.asarray(risks, dtype=np.uint32)
    n = len(keys)
    assert n > 0
    order = np.lexsort((keys, -risks.astype(np.int64)))
    labels = np.empty(n, dtype=object)
    c1 = int(math.ceil(0.01 * n))
    c5 = int(math.ceil(0.05 * n))
    c15 = int(math.ceil(0.15 * n))
    labels[order[:c1]] = "0_1"
    labels[order[c1:c5]] = "1_5"
    labels[order[c5:c15]] = "5_15"
    labels[order[c15:]] = "15_100"
    return labels


def region_score_stats_torch(logit_regions):
    """logit_regions: [N_regions, 32] -> [4, N_regions]."""
    mean = logit_regions.mean(dim=-1)
    maximum = logit_regions.max(dim=-1).values
    std = torch.sqrt(torch.clamp(((logit_regions - mean[:, None]) ** 2).mean(dim=-1), min=0.0))
    peakiness = maximum - mean
    return torch.stack((mean, maximum, std, peakiness), dim=0)


def regionalize_extended_orientation(feature_ch_len, presence_logit_len, *, reverse_region_axis):
    """
    feature_ch_len: [512, full_length]
    presence_logit_len: [full_length]
    Returns score_stats [4, 36], pooled [1024, 36].

    We pool the 2 left-neighbor + 32 target + 2 right-neighbor absolute R32 regions.
    For the RC orientation, within-region reversal does not affect mean/max/std/peakiness;
    only the 36-region axis is flipped back to genomic left->right.
    """
    start = profile_config.context_flank - NEIGHBOR_REGIONS * REGION_SIZE
    length = profile_config.target_length + 2 * NEIGHBOR_REGIONS * REGION_SIZE
    assert length == 36 * REGION_SIZE
    stop = start + length
    assert start >= 0 and stop <= feature_ch_len.shape[-1]

    feat = feature_ch_len[:, start:stop].reshape(512, 36, REGION_SIZE)
    pooled_mean = feat.mean(dim=-1)
    pooled_max = feat.max(dim=-1).values
    pooled = torch.cat((pooled_mean, pooled_max), dim=0)  # [1024, 36]

    logits = presence_logit_len[start:stop].reshape(36, REGION_SIZE)
    stats = region_score_stats_torch(logits)  # [4, 36]

    if reverse_region_axis:
        pooled = torch.flip(pooled, dims=[-1])
        stats = torch.flip(stats, dims=[-1])
    return stats, pooled


def regionalize_extended_orientation_cpu(feature_ch_len_gpu, presence_logit_len_gpu, *, reverse_region_axis):
    """
    Restart-safe DirectML path. Copy only the required 36x32 slice to CPU, then
    compute mean/max and score statistics with NumPy. Returns float32 NumPy arrays:
    score_stats [4,36], pooled [1024,36].
    """
    start = profile_config.context_flank - NEIGHBOR_REGIONS * REGION_SIZE
    length = profile_config.target_length + 2 * NEIGHBOR_REGIONS * REGION_SIZE
    assert length == 36 * REGION_SIZE
    stop = start + length
    assert start >= 0 and stop <= feature_ch_len_gpu.shape[-1]

    feat_cpu = (
        feature_ch_len_gpu[:, start:stop]
        .detach().cpu().numpy().astype(np.float32, copy=True)
    )
    logit_cpu = (
        presence_logit_len_gpu[start:stop]
        .detach().cpu().numpy().astype(np.float32, copy=True)
    )

    feat = feat_cpu.reshape(512, 36, REGION_SIZE)
    pooled_mean = feat.mean(axis=-1, dtype=np.float32)
    pooled_max = feat.max(axis=-1)
    pooled = np.concatenate((pooled_mean, pooled_max), axis=0).astype(np.float32, copy=False)

    logits = logit_cpu.reshape(36, REGION_SIZE)
    mean = logits.mean(axis=-1, dtype=np.float32)
    maximum = logits.max(axis=-1)
    centered = logits - mean[:, None]
    std = np.sqrt(np.maximum((centered * centered).mean(axis=-1, dtype=np.float32), 0.0))
    peakiness = maximum - mean
    stats = np.stack((mean, maximum, std, peakiness), axis=0).astype(np.float32, copy=False)

    if reverse_region_axis:
        pooled = pooled[:, ::-1].copy()
        stats = stats[:, ::-1].copy()

    del feat_cpu, logit_cpu, feat, logits, centered, pooled_mean, pooled_max, mean, maximum, std, peakiness
    return stats, pooled


def genomic_target_base_logit(presence_logit_len, *, reverse_base_axis):
    base = presence_logit_len[c:c+L]
    if reverse_base_axis:
        base = torch.flip(base, dims=[-1])
    return base


def natural_prior_logit(active_n, total_n):
    pi = float(active_n) / float(total_n)
    assert 0.0 < pi < 1.0
    return float(math.log(pi / (1.0 - pi)))


def tie_aware_recall_at_budget(total_hist, cohort_bins, budget):
    """Expected cohort recall under random ordering inside the 5dp cutoff tie."""
    total_hist = np.asarray(total_hist, dtype=np.int64)
    cohort_bins = np.asarray(cohort_bins, dtype=np.int64)
    budget = int(min(max(budget, 0), total_hist.sum()))
    if len(cohort_bins) == 0:
        return float("nan")
    if budget == 0:
        return 0.0
    desc = total_hist[::-1]
    cumulative = np.cumsum(desc, dtype=np.int64)
    idx = int(np.searchsorted(cumulative, budget, side="left"))
    cutoff_bin = SUBMISSION_LEVELS - 1 - idx
    before = int(cumulative[idx - 1]) if idx > 0 else 0
    tie_total = int(total_hist[cutoff_bin])
    take_from_tie = budget - before
    tie_fraction = float(take_from_tie / tie_total) if tie_total else 0.0
    above = int((cohort_bins > cutoff_bin).sum())
    at = int((cohort_bins == cutoff_bin).sum())
    return float((above + tie_fraction * at) / len(cohort_bins))

print("Helpers loaded.")


Helpers loaded.


In [4]:
# ============================================================
# ACCEPT / VERIFY COMPLETED STAGE 1 + STAGE 2 — DO NOT RECOMPUTE
# ============================================================
def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

POPULATION_PATH = RUN_DIR / "train_region_population.json"
HASH_SEAL_PATH = RUN_DIR / "EXP046B_recovery_input_hashes.json"
RECOVERY_INIT_PATH = RUN_DIR / "EXP046B_stage2_recovery_initialized.json"
STAGE2_COMPLETE_PATH = RUN_DIR / "EXP046B_stage2_cache_complete.json"
AMENDMENT_001_PATH = RUN_DIR / "EXP046B_preregistration_amendment_001.json"

# Exact recovery artifacts supplied after the successful Stage-2 recovery.
EXPECTED_RECOVERY_ARTIFACT_SHA256 = {
    "EXP046B_recovery_input_hashes.json": "f1aae14b81db8620a4c8d86bb265dc5a7394a9771c0af9a768dd2f9a8b02de45",
    "EXP046B_stage2_cache_complete.json": "0fca7db47a1d72a4852ca136b8888d89195ab50a66d0c657093ed0d88898d8ea",
    "EXP046B_stage2_recovery_initialized.json": "40562101783028713c603ef2a2111e5d12dbe1c3e1831474338a3767145b62b4",
    "EXP046B_preregistration_amendment_001.json": "bded12be6e4b7b8d8736ae9a50a1bc36803e82511b9c182044de0ed6838c5258",
    "sample_channel_sufficient_stats.npz": "391493cabf8f4afb7ad881d0dc449922cb39b010dbc726b920ea9e39e3753487",
}
for name, expected_sha in EXPECTED_RECOVERY_ARTIFACT_SHA256.items():
    p = RUN_DIR / name
    assert p.is_file(), f"Missing accepted recovery artifact: {p}"
    actual_sha = sha256_file(p)
    assert actual_sha == expected_sha, (name, actual_sha, expected_sha)

sealed = json.loads(HASH_SEAL_PATH.read_text(encoding="utf-8"))
for name, meta in sealed.items():
    p = RUN_DIR / name
    assert p.is_file(), p
    assert int(p.stat().st_size) == int(meta["size_bytes"]), name
    assert sha256_file(p) == meta["sha256"], name

stage2_complete = json.loads(STAGE2_COMPLETE_PATH.read_text(encoding="utf-8"))
assert stage2_complete == {
    "n_samples": 173887,
    "filled_count": 173887,
    "score_cache_bytes": 13910960,
    "pool_cache_bytes": 1780602880,
    "filled_cache_bytes": 173887,
    "cache_inference_batch_size": 1,
    "pooling": "CPU after limited 36x32 WIDTH512 feature slice copy",
    "data_first_journal_last": True,
}

sample_manifest = pd.read_csv(OUTPUTS["sample_manifest"])
sampling_summary = pd.read_csv(OUTPUTS["sampling_summary"])
train_lowtp = pd.read_csv(OUTPUTS["train_lowtp"])
population_payload = json.loads(POPULATION_PATH.read_text(encoding="utf-8"))
train_region_population = population_payload["per_contig"]

assert len(sample_manifest) == EXPECTED_TRAIN_SAMPLE_ROWS
assert int(sample_manifest["target"].sum()) == EXPECTED_TRAIN_SAMPLE_ACTIVE
assert int((sample_manifest["target"] == 0).sum()) == EXPECTED_TRAIN_SAMPLE_EMPTY
assert sample_manifest["region_key"].is_unique
assert sample_manifest["sample_row"].to_numpy(np.int64).tolist() == list(range(EXPECTED_TRAIN_SAMPLE_ROWS))
assert not sample_manifest.isna().any().any()
assert int(population_payload["total_valid_regions"]) == EXPECTED_TRAIN_VALID_REGIONS
assert int(population_payload["total_active_regions"]) == EXPECTED_TRAIN_ACTIVE_REGIONS
assert sum(int(v["empty_regions"]) for v in train_region_population.values()) == EXPECTED_TRAIN_EMPTY_REGIONS
assert len(train_lowtp) == EXPECTED_TRAIN_LOWTP
assert train_lowtp[["contig", "strand", "position"]].duplicated().sum() == 0

N_SAMPLES = len(sample_manifest)
score_context_path = CACHE_DIR / "score_context_f32.dat"
pool_context_path = CACHE_DIR / "backbone_meanmax_context_f16.dat"
filled_path = CACHE_DIR / "filled_u8.dat"
assert score_context_path.stat().st_size == stage2_complete["score_cache_bytes"]
assert pool_context_path.stat().st_size == stage2_complete["pool_cache_bytes"]
assert filled_path.stat().st_size == stage2_complete["filled_cache_bytes"]

# Stage 2 is complete: open read-only. V3 never writes this cache.
score_context_mm = np.memmap(
    score_context_path, mode="r", dtype=np.float32,
    shape=(N_SAMPLES, SCORE_STAT_CHANNELS, CONTEXT_REGIONS),
)
pool_context_mm = np.memmap(
    pool_context_path, mode="r", dtype=np.float16,
    shape=(N_SAMPLES, BACKBONE_POOL_CHANNELS, CONTEXT_REGIONS),
)
filled_mm = np.memmap(filled_path, mode="r", dtype=np.uint8, shape=(N_SAMPLES,))
assert int(np.asarray(filled_mm).sum()) == N_SAMPLES
assert np.all(np.asarray(filled_mm) == 1)

# Load the already-computed sufficient statistics instead of rescanning 1.67 GiB.
stats_npz = np.load(OUTPUTS["channel_stats"], allow_pickle=False)
stats_contigs = [str(x) for x in stats_npz["contigs"].tolist()]
assert stats_contigs == TRAIN_CONTIGS, (stats_contigs, TRAIN_CONTIGS)
per_contig_sum = stats_npz["channel_sum"].astype(np.float64, copy=True)
per_contig_sumsq = stats_npz["channel_sumsq"].astype(np.float64, copy=True)
per_contig_count = stats_npz["channel_count"].astype(np.int64, copy=True)
stats_npz.close()
assert per_contig_sum.shape == (len(TRAIN_CONTIGS), VARIANT_CHANNELS["B"])
assert per_contig_sumsq.shape == per_contig_sum.shape
assert per_contig_count.shape == (len(TRAIN_CONTIGS),)
assert np.isfinite(per_contig_sum).all() and np.isfinite(per_contig_sumsq).all()
assert np.all(per_contig_count > 0)
contig_to_train_index = {contig: i for i, contig in enumerate(TRAIN_CONTIGS)}

def scaler_for_training_contigs(training_contigs, variant):
    ids = [contig_to_train_index[c] for c in training_contigs]
    s = per_contig_sum[ids].sum(axis=0)
    ss = per_contig_sumsq[ids].sum(axis=0)
    n = int(per_contig_count[ids].sum())
    mean = s / n
    var = np.maximum(ss / n - mean * mean, 1e-8)
    std = np.sqrt(var)
    channels = VARIANT_CHANNELS[variant]
    mean = mean[:channels].astype(np.float32)
    std = std[:channels].astype(np.float32)
    assert np.isfinite(mean).all() and np.isfinite(std).all() and np.all(std > 0)
    return mean, std

print("Accepted Stage 1/2 state: PASS")
print("  samples:", f"{N_SAMPLES:,}")
print("  Stage-2 cache complete:", STAGE2_COMPLETE_PATH)
print("  sufficient stats SHA256:", EXPECTED_RECOVERY_ARTIFACT_SHA256["sample_channel_sufficient_stats.npz"])


Accepted Stage 1/2 state: PASS
  samples: 173,887
  Stage-2 cache complete: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32\run_001\EXP046B_stage2_cache_complete.json
  sufficient stats SHA256: 391493cabf8f4afb7ad881d0dc449922cb39b010dbc726b920ea9e39e3753487


In [5]:
# ============================================================
# PREREGISTRATION AMENDMENT 002 — RUNTIME RECOVERY BEFORE ANY CV AP
# ============================================================
# We observed only fold-1 TRAIN losses before the DML kernel restart.
# No held-out contig AP, no pooled CV metric, no alpha selection and no validation result existed.
AMENDMENT_002_PATH = RUN_DIR / "EXP046B_preregistration_amendment_002.json"
legacy_dml_models = sorted(
    p.name for p in FOLD_MODEL_DIR.glob("fold_01_*_?.pt") if p.is_file()
)

amendment_002 = {
    "experiment_id": EXPERIMENT_ID,
    "run": "run_001",
    "amendment_id": "002",
    "timing": (
        "after fold-1 A/B train-loss logs were printed and the kernel restarted; "
        "before any held-out-contig CV AP, pooled CV result, alpha/config selection, or validation evaluation"
    ),
    "reason": (
        "DirectML BCEWithLogitsLoss emitted aten::log_sigmoid_forward CPU fallback; "
        "kernel restarted immediately after fold-1 training when entering genome-wide CV inference. "
        "Make the small regional head CPU-only and make held-out-contig evaluation restart-safe."
    ),
    "observed_before_amendment": {
        "fold1_train_loss_A_epoch4": 0.369682,
        "fold1_train_loss_B_epoch4": 0.343045,
        "heldout_cv_ap_seen": False,
        "pooled_cv_seen": False,
        "validation_used": False,
    },
    "scientific_design_unchanged": [
        "same run_001 and accepted Stage-1 sample manifest",
        "same R32 absolute strand-specific grid",
        "same frozen EXP045 WIDTH512 backbone and nucleotide head",
        "same A/B architectures and inputs",
        "same class-balanced IPW regional BCE objective",
        "same four-epoch 5% warm-up plus cosine schedule from amendment 001",
        "same alpha grid",
        "same leave-one-train-contig-out CV selection metric: pooled natural nucleotide AP5",
        "validation remains untouched until one config is frozen",
    ],
    "runtime_changes": {
        "regional_head_training_device": "CPU",
        "regional_head_inference_device": "CPU",
        "optimizer_class": "DirectMLAdam retained; parameters/tensors are CPU",
        "backbone_inference_device": "DirectML",
        "legacy_DML_fold1_models": "preserved on disk but excluded from all V3 CV metrics",
        "cv_evaluator": "atomic sharded evaluator, 16 tiles per shard; completed shards are immutable restart journal",
        "partial_shard_policy": "temporary file is never counted; at most current 16-tile shard is recomputed after restart",
    },
}
if AMENDMENT_002_PATH.exists():
    existing = json.loads(AMENDMENT_002_PATH.read_text(encoding="utf-8"))
    assert existing == amendment_002, "Amendment 002 differs from frozen copy on disk"
else:
    tmp = AMENDMENT_002_PATH.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(amendment_002, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, AMENDMENT_002_PATH)

print("Amendment 002 frozen:", AMENDMENT_002_PATH)
print("Legacy DML fold-1 models preserved/excluded:", legacy_dml_models or "none found")
print("Validation still untouched by EXP046-B: PASS")


Amendment 002 frozen: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32\run_001\EXP046B_preregistration_amendment_002.json
Legacy DML fold-1 models preserved/excluded: ['fold_01_NC_064035.1_A.pt', 'fold_01_NC_064035.1_B.pt']
Validation still untouched by EXP046-B: PASS


## Stage 3 — CPU heads, same frozen scientific design

The head architecture, sampling, IPW loss, LR schedule, alpha grid and leave-one-contig-out selection rule are unchanged. Only the execution device of the small head changes from DirectML to CPU to eliminate the observed BCE fallback path.


In [6]:
# ============================================================
# CPU REGIONAL HEAD / TRAINING HELPERS
# Frozen WIDTH512 backbone remains on DirectML; only the tiny head is CPU-only.
# ============================================================
HEAD_DEVICE = torch.device("cpu")

class RegionalCorrectionHead(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 64, kernel_size=1)
        self.act = nn.GELU()
        self.conv2 = nn.Conv1d(64, 1, kernel_size=5, padding=0)

    def forward(self, x):
        y = self.conv2(self.act(self.conv1(x)))
        assert y.shape[-1] == 1
        return y[:, 0, 0]


def make_optimizer_cpu(head):
    # Retain the exact optimizer class used by the preregistered notebook; only tensors live on CPU.
    sig = inspect.signature(DirectMLAdam)
    if "weight_decay" in sig.parameters:
        return DirectMLAdam(head.parameters(), lr=HEAD_PEAK_LR, weight_decay=HEAD_WEIGHT_DECAY), False
    return DirectMLAdam(head.parameters(), lr=HEAD_PEAK_LR), True


def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group["lr"] = float(lr)


def lr_for_optimizer_step(step_index, total_steps):
    assert 0 <= step_index < total_steps
    warmup_steps = max(1, int(math.ceil(HEAD_WARMUP_FRACTION * total_steps)))
    if step_index < warmup_steps:
        return float(HEAD_PEAK_LR * (step_index + 1) / warmup_steps)
    decay_steps = total_steps - warmup_steps
    if decay_steps <= 1:
        return float(HEAD_FINAL_LR)
    decay_index = step_index - warmup_steps
    progress = decay_index / (decay_steps - 1)
    return float(
        HEAD_FINAL_LR
        + 0.5 * (HEAD_PEAK_LR - HEAD_FINAL_LR) * (1.0 + math.cos(math.pi * progress))
    )


def count_mixed_optimizer_steps(y_all):
    rng = np.random.default_rng(HEAD_SEED)
    total = 0
    for _epoch in range(HEAD_EPOCHS):
        order = rng.permutation(len(y_all))
        for bs in range(0, len(order), HEAD_BATCH_SIZE):
            loc = order[bs:bs + HEAD_BATCH_SIZE]
            y_np = y_all[loc]
            if len(y_np) and not np.all(y_np == y_np[0]):
                total += 1
    assert total > 0
    return total


def load_sample_batch(sample_rows, variant, mean, std):
    sample_rows = np.asarray(sample_rows, dtype=np.int64)
    score = np.asarray(score_context_mm[sample_rows], dtype=np.float32)
    if variant == "A":
        x = score
    else:
        pool = np.asarray(pool_context_mm[sample_rows], dtype=np.float32)
        x = np.concatenate((score, pool), axis=1)
    x = (x - mean[None, :, None]) / std[None, :, None]
    return torch.from_numpy(np.ascontiguousarray(x.astype(np.float32, copy=False)))


def balanced_ipw_bce(logit, target, importance_weight):
    # Runs entirely on CPU in V3; no DML aten::log_sigmoid_forward fallback.
    per = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    pos = target > 0.5
    neg = ~pos
    assert bool(pos.any()) and bool(neg.any())
    pos_loss = per[pos].mean()
    neg_w = importance_weight[neg]
    neg_loss = (per[neg] * neg_w).sum() / torch.clamp(neg_w.sum(), min=1e-12)
    return 0.5 * (pos_loss + neg_loss)


def atomic_torch_save(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def atomic_csv(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def train_head_cpu(training_contigs, variant, fold_name):
    training_contigs = list(training_contigs)
    rows = sample_manifest.loc[
        sample_manifest["contig"].astype(str).isin(training_contigs), "sample_row"
    ].to_numpy(np.int64)
    y_all = sample_manifest.loc[rows, "target"].to_numpy(np.float32)
    w_all = sample_manifest.loc[rows, "importance_weight"].to_numpy(np.float32)
    assert int((y_all == 1).sum()) > 0 and int((y_all == 0).sum()) > 0

    mean, std = scaler_for_training_contigs(training_contigs, variant)
    torch.manual_seed(HEAD_SEED)
    head = RegionalCorrectionHead(VARIANT_CHANNELS[variant]).to(HEAD_DEVICE)
    optimizer, manual_wd = make_optimizer_cpu(head)
    total_optimizer_steps = count_mixed_optimizer_steps(y_all)
    warmup_steps = max(1, int(math.ceil(HEAD_WARMUP_FRACTION * total_optimizer_steps)))
    rng = np.random.default_rng(HEAD_SEED)
    history = []
    optimizer_step = 0

    head.train()
    for epoch in range(1, HEAD_EPOCHS + 1):
        order = rng.permutation(len(rows))
        epoch_loss_num = 0.0
        epoch_batches = 0
        epoch_first_lr = None
        epoch_last_lr = None
        for bs in range(0, len(order), HEAD_BATCH_SIZE):
            loc = order[bs:bs + HEAD_BATCH_SIZE]
            y_np = y_all[loc]
            if np.all(y_np == y_np[0]):
                continue
            current_lr = lr_for_optimizer_step(optimizer_step, total_optimizer_steps)
            set_optimizer_lr(optimizer, current_lr)
            epoch_first_lr = current_lr if epoch_first_lr is None else epoch_first_lr
            epoch_last_lr = current_lr

            sample_rows = rows[loc]
            x = load_sample_batch(sample_rows, variant, mean, std)  # CPU
            y = torch.from_numpy(y_np)                              # CPU
            iw = torch.from_numpy(w_all[loc])                       # CPU

            optimizer.zero_grad(set_to_none=True)
            logit = head(x)
            loss = balanced_ipw_bce(logit, y, iw)
            loss.backward()
            optimizer.step()
            if manual_wd and HEAD_WEIGHT_DECAY > 0:
                with torch.no_grad():
                    decay = 1.0 - current_lr * HEAD_WEIGHT_DECAY
                    for p in head.parameters():
                        p.mul_(decay)

            epoch_loss_num += float(loss.detach().item())
            epoch_batches += 1
            optimizer_step += 1
            del x, y, iw, logit, loss

        mean_loss = epoch_loss_num / max(epoch_batches, 1)
        history.append({
            "fold": fold_name,
            "variant": variant,
            "epoch": epoch,
            "loss": mean_loss,
            "optimizer_steps_completed": optimizer_step,
            "optimizer_steps_total": total_optimizer_steps,
            "warmup_steps": warmup_steps,
            "epoch_first_lr": epoch_first_lr,
            "epoch_last_lr": epoch_last_lr,
            "device": "cpu",
        })
        print(
            f"[{fold_name}][{variant}][CPU] epoch {epoch}/{HEAD_EPOCHS} loss={mean_loss:.6f} "
            f"lr={epoch_first_lr:.3g}->{epoch_last_lr:.3g} step={optimizer_step}/{total_optimizer_steps}"
        )

    assert optimizer_step == total_optimizer_steps
    assert abs(lr_for_optimizer_step(total_optimizer_steps - 1, total_optimizer_steps) - HEAD_FINAL_LR) <= 1e-15

    active_n = sum(train_region_population[c]["active_regions"] for c in training_contigs)
    total_n = sum(train_region_population[c]["valid_regions"] for c in training_contigs)
    prior_offset = natural_prior_logit(active_n, total_n)
    head.eval()
    return head, mean, std, prior_offset, pd.DataFrame(history)


def head_region_logits_from_extended_cpu(stats_4x36, pooled_1024x36, head, variant, mean, std, prior_offset):
    if variant == "A":
        x_np = stats_4x36
    else:
        # Match f16 training-cache quantization before head inference.
        pool_q = pooled_1024x36.astype(np.float16).astype(np.float32)
        x_np = np.concatenate((stats_4x36, pool_q), axis=0)
    x_np = (x_np - mean[:, None]) / std[:, None]
    x = torch.from_numpy(np.ascontiguousarray(x_np[None].astype(np.float32)))
    assert next(head.parameters()).device.type == "cpu"
    with torch.no_grad():
        raw = head.conv2(head.act(head.conv1(x)))[:, 0, :]
    g = raw[0].numpy().astype(np.float32, copy=True) + np.float32(prior_offset)
    del x, raw
    assert g.shape == (REGIONS_PER_TILE,)
    return g


def fold_model_paths(fold_i, heldout, variant):
    stem = f"fold_{fold_i:02d}_{heldout}_{variant}_cpu_v3"
    return CPU_FOLD_MODEL_DIR / f"{stem}.pt", CPU_FOLD_MODEL_DIR / f"{stem}_history.csv"


def ensure_cpu_fold_model(fold_i, heldout, training_contigs, variant):
    model_path, history_path = fold_model_paths(fold_i, heldout, variant)
    if model_path.is_file() and history_path.is_file():
        package = torch.load(model_path, map_location="cpu", weights_only=False)
        assert package["heldout_contig"] == heldout
        assert package["variant"] == variant
        assert package["head_device"] == "cpu"
        assert package["training_schedule"] == TRAINING_SCHEDULE
        head = RegionalCorrectionHead(VARIANT_CHANNELS[variant]).to(HEAD_DEVICE)
        head.load_state_dict(package["state_dict"])
        head.eval()
        print(f"[{heldout}][{variant}] CPU model already durable — reuse")
        return head, package["scaler_mean"], package["scaler_std"], float(package["prior_offset"]), pd.read_csv(history_path)

    assert not model_path.exists() and not history_path.exists(), (
        "Partial CPU fold-model artifact exists; inspect before continuing", model_path, history_path
    )
    head, mean, std, prior, history = train_head_cpu(training_contigs, variant, heldout)
    atomic_csv(history, history_path)
    payload = {
        "experiment_id": EXPERIMENT_ID,
        "amendment": "002",
        "heldout_contig": heldout,
        "training_contigs": list(training_contigs),
        "variant": variant,
        "head_device": "cpu",
        "state_dict": {k: v.detach().cpu() for k, v in head.state_dict().items()},
        "scaler_mean": mean,
        "scaler_std": std,
        "prior_offset": prior,
        "training_schedule": TRAINING_SCHEDULE,
        "weight_decay": HEAD_WEIGHT_DECAY,
        "source_cache_sha256": EXPECTED_RECOVERY_ARTIFACT_SHA256["sample_channel_sufficient_stats.npz"],
    }
    atomic_torch_save(payload, model_path)
    print(f"[{heldout}][{variant}] CPU model durable: {model_path.name}")
    return head, mean, std, prior, history

# Scheduler smoke only.
for _n in (20, 101, 1000):
    _lr = np.asarray([lr_for_optimizer_step(i, _n) for i in range(_n)], dtype=np.float64)
    _w = max(1, int(math.ceil(HEAD_WARMUP_FRACTION * _n)))
    assert np.isfinite(_lr).all()
    assert abs(_lr[-1] - HEAD_FINAL_LR) <= 1e-15
    assert _lr.max() <= HEAD_PEAK_LR + 1e-15
    assert np.all(np.diff(_lr[_w - 1:]) <= 1e-15)

# CPU optimizer/loss smoke on synthetic data — fails before any fold training if DirectMLAdam is not CPU-compatible.
_smoke_head = RegionalCorrectionHead(SCORE_STAT_CHANNELS).to(HEAD_DEVICE)
_smoke_opt, _smoke_manual_wd = make_optimizer_cpu(_smoke_head)
_smoke_x = torch.zeros((4, SCORE_STAT_CHANNELS, CONTEXT_REGIONS), dtype=torch.float32)
_smoke_y = torch.tensor([0.0, 1.0, 0.0, 1.0], dtype=torch.float32)
_smoke_w = torch.ones(4, dtype=torch.float32)
_smoke_opt.zero_grad(set_to_none=True)
_smoke_loss = balanced_ipw_bce(_smoke_head(_smoke_x), _smoke_y, _smoke_w)
_smoke_loss.backward()
_smoke_opt.step()
assert all(p.device.type == "cpu" and torch.isfinite(p).all() for p in _smoke_head.parameters())
del _smoke_head, _smoke_opt, _smoke_x, _smoke_y, _smoke_w, _smoke_loss
gc.collect()
print("CPU DirectMLAdam/BCE smoke: PASS")

print("CPU head helpers loaded. WIDTH512 backbone remains DirectML-only.")


CPU DirectMLAdam/BCE smoke: PASS
CPU head helpers loaded. WIDTH512 backbone remains DirectML-only.


## Restart-safe CV evaluator

A held-out contig is evaluated in immutable 16-tile shards. Each shard is written to a temporary file, flushed+fsynced, atomically renamed, then verified. Existing contiguous shards are reused after a restart; an incomplete temporary shard is discarded.


In [7]:
# ============================================================
# RESTART-SAFE SHARDED CV EVALUATOR
# One atomic shard = at most 16 held-out tiles. Existing complete shards are immutable.
# ============================================================
def atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        np.savez(f, **arrays)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def alpha_tag(alpha):
    return str(float(alpha)).replace("-", "m").replace(".", "p")


def empty_danger_strata(keys, risks, active):
    keys = np.asarray(keys, dtype=np.uint64)
    risks = np.asarray(risks, dtype=np.uint32)
    active = np.asarray(active, dtype=bool)
    labels = np.full(len(keys), "ACTIVE", dtype=object)
    empty_idx = np.flatnonzero(~active)
    labels[empty_idx] = rank_strata_from_risk(keys[empty_idx], risks[empty_idx])
    return labels


def regional_diagnostics(contig, variant, region_g, region_active, region_risk, region_key, phase):
    region_g = np.asarray(region_g, dtype=np.float32)
    region_active = np.asarray(region_active, dtype=bool)
    region_risk = np.asarray(region_risk, dtype=np.uint32)
    region_key = np.asarray(region_key, dtype=np.uint64)
    strata = empty_danger_strata(region_key, region_risk, region_active)
    rows = []
    auc = float(roc_auc_score(region_active.astype(np.uint8), region_g))
    active_g = np.sort(region_g[region_active])
    for recall_target in REGIONAL_RECALL_DIAGNOSTIC_TARGETS:
        drop_active_n = int(math.floor((1.0 - recall_target) * len(active_g)))
        threshold = -np.inf if drop_active_n <= 0 else float(active_g[min(drop_active_n - 1, len(active_g) - 1)])
        suppressed = region_g <= threshold
        row = {
            "phase": phase,
            "contig": contig,
            "variant": variant,
            "regional_auc": auc,
            "target_active_recall": recall_target,
            "realized_active_recall": float((~suppressed[region_active]).mean()),
            "suppressed_empty_all": float(suppressed[~region_active].mean()),
        }
        for stratum in NEGATIVE_STRATA:
            m = strata == stratum
            row[f"suppressed_empty_{stratum}"] = float(suppressed[m].mean()) if int(m.sum()) else float("nan")
        rows.append(row)
    return rows


def fold_eval_paths(fold_i, contig):
    fold_dir = CV_V3_DIR / f"fold_{fold_i:02d}_{contig}"
    shard_dir = fold_dir / "shards"
    fold_dir.mkdir(exist_ok=True)
    shard_dir.mkdir(exist_ok=True)
    return {
        "dir": fold_dir,
        "shards": shard_dir,
        "manifest": fold_dir / "fold_eval_manifest.json",
        "metrics": fold_dir / "fold_metrics.csv",
        "regional": fold_dir / "regional_diagnostics.csv",
        "hist": fold_dir / "fold_histograms_and_lowtp.npz",
        "complete": fold_dir / "fold_complete.json",
    }


def shard_path(shard_dir, start_tile, end_tile):
    return shard_dir / f"tiles_{start_tile:06d}_{end_tile:06d}.npz"


def verify_existing_shards(paths, contig, n_tiles, model_sha_A, model_sha_B):
    # Incomplete temp files are never scientific state.
    for tmp in paths["shards"].glob("*.tmp"):
        tmp.unlink()
    manifest_payload = {
        "format_version": 3,
        "contig": contig,
        "n_tiles": int(n_tiles),
        "shard_tiles": int(CV_SHARD_TILES),
        "model_sha_A": model_sha_A,
        "model_sha_B": model_sha_B,
        "backbone_checkpoint": str(CHECKPOINT_PATH),
        "amendment": "002",
    }
    if paths["manifest"].exists():
        existing = json.loads(paths["manifest"].read_text(encoding="utf-8"))
        assert existing == manifest_payload, "Fold evaluator manifest/model hashes changed"
    else:
        atomic_json(paths["manifest"], manifest_payload)

    expected_start = 0
    complete_shards = []
    while expected_start < n_tiles:
        expected_end = min(expected_start + CV_SHARD_TILES, n_tiles)
        p = shard_path(paths["shards"], expected_start, expected_end)
        if not p.exists():
            break
        with np.load(p, allow_pickle=False) as d:
            assert int(d["format_version"]) == 3
            assert str(d["contig"].item()) == contig
            assert int(d["tile_start"]) == expected_start
            assert int(d["tile_end"]) == expected_end
            assert str(d["model_sha_A"].item()) == model_sha_A
            assert str(d["model_sha_B"].item()) == model_sha_B
            n = len(d["region_key"])
            for name in ("region_risk5", "region_active", "g_A", "g_B", "base_logit", "valid_mask", "positive_mask"):
                assert len(d[name]) == n, (p, name)
            assert d["base_logit"].shape == (n, REGION_SIZE)
            assert np.isfinite(d["g_A"]).all() and np.isfinite(d["g_B"]).all() and np.isfinite(d["base_logit"]).all()
        complete_shards.append(p)
        expected_start = expected_end

    # No out-of-order committed shards may exist beyond the contiguous journal frontier.
    expected_names = {p.name for p in complete_shards}
    extras = [p for p in paths["shards"].glob("tiles_*.npz") if p.name not in expected_names]
    assert not extras, f"Non-contiguous committed shards found; inspect manually: {extras[:3]}"
    return complete_shards, expected_start


def evaluate_fold_to_atomic_shards(
    fold_i, contig,
    head_A, mean_A, std_A, prior_A,
    head_B, mean_B, std_B, prior_B,
    model_path_A, model_path_B,
):
    contig_tiles = train_tiles[train_tiles["contig"].astype(str) == contig].reset_index(drop=True)
    contig_code = CONTIG_TO_CODE[contig]
    paths = fold_eval_paths(fold_i, contig)
    sha_A = sha256_file(model_path_A)
    sha_B = sha256_file(model_path_B)
    shards, resume_tile = verify_existing_shards(paths, contig, len(contig_tiles), sha_A, sha_B)

    if resume_tile == len(contig_tiles):
        print(f"[{contig}] all {len(shards)} CV shards already durable — finalize/reuse")
        return paths

    print(
        f"[{contig}] restart-safe CV inference resumes at tile {resume_tile:,}/{len(contig_tiles):,}; "
        f"durable_shards={len(shards)}"
    )
    started = time.perf_counter()
    with torch.no_grad():
        for chunk_start in range(resume_tile, len(contig_tiles), CV_SHARD_TILES):
            chunk_end = min(chunk_start + CV_SHARD_TILES, len(contig_tiles))
            region_keys = []
            region_risks = []
            region_active = []
            region_g_A = []
            region_g_B = []
            raw_base_regions = []
            raw_valid_masks = []
            raw_positive_masks = []

            for tile_i in range(chunk_start, chunk_end):
                selected = contig_tiles.iloc[tile_i:tile_i + 1]
                batch = train_generator.make_batch(selected)
                assert len(selected) == EVAL_INFERENCE_BATCH_SIZE == 1
                x_gpu = batch.x_both_strands.to(device)
                feature_gpu = model.backbone(x_gpu)
                output_gpu = model.profile_head(feature_gpu)

                row = selected.iloc[0]
                target_start = int(row["target_start"])
                first_region_id = target_start // REGION_SIZE
                target_region_ids = first_region_id + np.arange(REGIONS_PER_TILE, dtype=np.int64)
                plus_count = batch.count[0, 0, :].numpy().astype(np.float32, copy=False)
                minus_count = batch.count[0, 1, :].numpy().astype(np.float32, copy=False)
                plus_mask = batch.mask[0, 0, :].numpy().astype(bool, copy=False)
                minus_mask = batch.mask[0, 1, :].numpy().astype(bool, copy=False)

                for strand_code, orient_index, reverse_axis, count, mask in (
                    (0, 0, False, plus_count, plus_mask),
                    (1, 1, True, minus_count, minus_mask),
                ):
                    stats, pool = regionalize_extended_orientation_cpu(
                        feature_gpu[orient_index], output_gpu[orient_index, 0], reverse_region_axis=reverse_axis
                    )
                    base_logit = genomic_target_base_logit(
                        output_gpu[orient_index, 0], reverse_base_axis=reverse_axis
                    ).detach().cpu().numpy().astype(np.float32, copy=True)

                    g_A = head_region_logits_from_extended_cpu(stats, pool, head_A, "A", mean_A, std_A, prior_A)
                    g_B = head_region_logits_from_extended_cpu(stats, pool, head_B, "B", mean_B, std_B, prior_B)

                    valid_2d = mask.reshape(REGIONS_PER_TILE, REGION_SIZE)
                    positive_2d = ((count > 0) & mask).reshape(REGIONS_PER_TILE, REGION_SIZE)
                    valid_regions = valid_2d.any(axis=1)
                    base_2d = base_logit.reshape(REGIONS_PER_TILE, REGION_SIZE)
                    base_bins_2d = probability_bin_5dp_from_logit(base_2d)
                    risk = np.where(valid_2d, base_bins_2d, np.uint32(0)).max(axis=1).astype(np.uint32)
                    active = positive_2d.any(axis=1)
                    keys = pack_region_keys(contig_code, strand_code, target_region_ids)

                    region_keys.append(keys[valid_regions])
                    region_risks.append(risk[valid_regions])
                    region_active.append(active[valid_regions])
                    region_g_A.append(g_A[valid_regions])
                    region_g_B.append(g_B[valid_regions])
                    raw_base_regions.append(base_2d[valid_regions].copy())
                    raw_valid_masks.append(bool_rows_to_u32(valid_2d[valid_regions]))
                    raw_positive_masks.append(bool_rows_to_u32(positive_2d[valid_regions]))
                    del stats, pool, base_logit, g_A, g_B, valid_2d, positive_2d, base_2d, base_bins_2d

                del x_gpu, feature_gpu, output_gpu, batch, selected

            key_arr = np.concatenate(region_keys).astype(np.uint64, copy=False)
            risk_arr = np.concatenate(region_risks).astype(np.uint32, copy=False)
            active_arr = np.concatenate(region_active).astype(np.uint8, copy=False)
            gA_arr = np.concatenate(region_g_A).astype(np.float32, copy=False)
            gB_arr = np.concatenate(region_g_B).astype(np.float32, copy=False)
            base_arr = np.concatenate(raw_base_regions).astype(np.float32, copy=False)
            valid_mask_arr = np.concatenate(raw_valid_masks).astype(np.uint32, copy=False)
            pos_mask_arr = np.concatenate(raw_positive_masks).astype(np.uint32, copy=False)
            assert len(np.unique(key_arr)) == len(key_arr)

            p = shard_path(paths["shards"], chunk_start, chunk_end)
            atomic_npz(
                p,
                format_version=np.asarray(3, dtype=np.int32),
                contig=np.asarray(contig),
                tile_start=np.asarray(chunk_start, dtype=np.int64),
                tile_end=np.asarray(chunk_end, dtype=np.int64),
                model_sha_A=np.asarray(sha_A),
                model_sha_B=np.asarray(sha_B),
                region_key=key_arr,
                region_risk5=risk_arr,
                region_active=active_arr,
                g_A=gA_arr,
                g_B=gB_arr,
                base_logit=base_arr,
                valid_mask=valid_mask_arr,
                positive_mask=pos_mask_arr,
            )
            # Verify the just-committed shard before advancing the journal frontier.
            with np.load(p, allow_pickle=False) as check:
                assert int(check["tile_start"]) == chunk_start and int(check["tile_end"]) == chunk_end
                assert np.isfinite(check["base_logit"]).all() and np.isfinite(check["g_A"]).all() and np.isfinite(check["g_B"]).all()

            del region_keys, region_risks, region_active, region_g_A, region_g_B
            del raw_base_regions, raw_valid_masks, raw_positive_masks
            del key_arr, risk_arr, active_arr, gA_arr, gB_arr, base_arr, valid_mask_arr, pos_mask_arr
            gc.collect()
            print(
                f"[{contig}] durable CV shard tiles {chunk_start:,}:{chunk_end:,} / {len(contig_tiles):,} "
                f"elapsed={(time.perf_counter()-started)/60:.1f} min"
            )

    shards, resume_tile = verify_existing_shards(paths, contig, len(contig_tiles), sha_A, sha_B)
    assert resume_tile == len(contig_tiles)
    print(f"[{contig}] sharded inference COMPLETE — {len(shards)} durable shards")
    return paths


def lowtp_lookup_for_contig(contig):
    contig_code = CONTIG_TO_CODE[contig]
    d = train_lowtp[train_lowtp["contig"].astype(str) == contig]
    lookup = {}
    for row in d.itertuples(index=False):
        strand_code = 0 if str(row.strand) == "+" else 1
        pos = int(row.position)
        key = int(pack_region_keys(contig_code, strand_code, np.asarray([pos // REGION_SIZE], dtype=np.int64))[0])
        lookup.setdefault(key, []).append(pos % REGION_SIZE)
    assert sum(len(v) for v in lookup.values()) == len(d)
    return d, lookup


def finalize_fold_from_shards(fold_i, contig, paths):
    if paths["complete"].is_file():
        payload = json.loads(paths["complete"].read_text(encoding="utf-8"))
        for name in ("metrics", "regional", "hist"):
            p = paths[name]
            assert p.is_file() and sha256_file(p) == payload["outputs_sha256"][p.name]
        print(f"[{contig}] fold result already finalized — reuse")
        return

    contig_tiles = train_tiles[train_tiles["contig"].astype(str) == contig].reset_index(drop=True)
    model_path_A, _ = fold_model_paths(fold_i, contig, "A")
    model_path_B, _ = fold_model_paths(fold_i, contig, "B")
    sha_A, sha_B = sha256_file(model_path_A), sha256_file(model_path_B)
    shards, frontier = verify_existing_shards(paths, contig, len(contig_tiles), sha_A, sha_B)
    assert frontier == len(contig_tiles)

    hist_total = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}
    hist_pos = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}
    lowtp_bins = {(v, a): [] for v in ("A", "B") for a in ALPHAS}
    lowtp_contig, lowtp_lookup = lowtp_lookup_for_contig(contig)
    region_keys, region_risks, region_active, region_g_A, region_g_B = [], [], [], [], []
    positions_seen = 0
    positives_seen = 0

    for p in shards:
        with np.load(p, allow_pickle=False) as d:
            keys = d["region_key"].astype(np.uint64, copy=False)
            risk = d["region_risk5"].astype(np.uint32, copy=False)
            active = d["region_active"].astype(bool, copy=False)
            gA = d["g_A"].astype(np.float32, copy=False)
            gB = d["g_B"].astype(np.float32, copy=False)
            base = d["base_logit"].astype(np.float32, copy=False)
            valid = u32_to_bool_rows(d["valid_mask"])
            positive = u32_to_bool_rows(d["positive_mask"])
            positions_seen += int(valid.sum())
            positives_seen += int(positive.sum())

            key_to_row = {int(k): i for i, k in enumerate(keys)}
            for variant, g in (("A", gA), ("B", gB)):
                for alpha in ALPHAS:
                    bins = probability_bin_5dp_from_logit(base + np.float32(alpha) * g[:, None])
                    update_histogram(hist_total[(variant, alpha)], bins[valid])
                    update_histogram(hist_pos[(variant, alpha)], bins[positive])
                    for key, offsets in lowtp_lookup.items():
                        j = key_to_row.get(key)
                        if j is not None:
                            lowtp_bins[(variant, alpha)].extend(bins[j, offsets].astype(np.int64).tolist())

            region_keys.append(keys.copy())
            region_risks.append(risk.copy())
            region_active.append(active.copy())
            region_g_A.append(gA.copy())
            region_g_B.append(gB.copy())

    assert positions_seen == int(contig_tiles["allowed_position_strand"].sum()), (contig, positions_seen)
    assert positives_seen > 0
    assert all(len(lowtp_bins[(v, a)]) == len(lowtp_contig) for v in ("A", "B") for a in ALPHAS)

    key_arr = np.concatenate(region_keys).astype(np.uint64, copy=False)
    risk_arr = np.concatenate(region_risks).astype(np.uint32, copy=False)
    active_arr = np.concatenate(region_active).astype(bool, copy=False)
    gA_arr = np.concatenate(region_g_A).astype(np.float32, copy=False)
    gB_arr = np.concatenate(region_g_B).astype(np.float32, copy=False)
    assert len(np.unique(key_arr)) == len(key_arr)

    fold_metrics = []
    for variant in ("A", "B"):
        for alpha in ALPHAS:
            fold_metrics.append({
                "heldout_contig": contig,
                "variant": variant,
                "alpha": alpha,
                "ap5": ap_from_histograms(hist_pos[(variant, alpha)], hist_total[(variant, alpha)]),
                "positive_n": int(hist_pos[(variant, alpha)].sum()),
                "position_n": int(hist_total[(variant, alpha)].sum()),
                "lowtp_n": len(lowtp_bins[(variant, alpha)]),
            })
    metrics_df = pd.DataFrame(fold_metrics)
    regional_rows = []
    regional_rows += regional_diagnostics(contig, "A", gA_arr, active_arr, risk_arr, key_arr, "train_cv")
    regional_rows += regional_diagnostics(contig, "B", gB_arr, active_arr, risk_arr, key_arr, "train_cv")
    regional_df = pd.DataFrame(regional_rows)

    hist_payload = {}
    for variant in ("A", "B"):
        for alpha in ALPHAS:
            tag = alpha_tag(alpha)
            hist_payload[f"total_{variant}_{tag}"] = hist_total[(variant, alpha)]
            hist_payload[f"pos_{variant}_{tag}"] = hist_pos[(variant, alpha)]
            hist_payload[f"lowtp_{variant}_{tag}"] = np.asarray(lowtp_bins[(variant, alpha)], dtype=np.uint32)
    atomic_csv(metrics_df, paths["metrics"])
    atomic_csv(regional_df, paths["regional"])
    atomic_npz(paths["hist"], **hist_payload)

    output_hashes = {p.name: sha256_file(p) for p in (paths["metrics"], paths["regional"], paths["hist"])}
    complete = {
        "format_version": 3,
        "fold": int(fold_i),
        "heldout_contig": contig,
        "positions_seen": int(positions_seen),
        "positives_seen": int(positives_seen),
        "fixed_lowtp_seen_per_config": int(len(lowtp_contig)),
        "n_shards": int(len(shards)),
        "model_sha_A": sha_A,
        "model_sha_B": sha_B,
        "outputs_sha256": output_hashes,
    }
    atomic_json(paths["complete"], complete)
    print(f"[{contig}] fold FINALIZED — positions={positions_seen:,}, positives={positives_seen:,}")

print("Restart-safe CV evaluator loaded. Shard size:", CV_SHARD_TILES, "tiles")


Restart-safe CV evaluator loaded. Shard size: 16 tiles


In [ ]:
# ============================================================
# RUN / RESUME 9-FOLD TRAIN-CONTIG CV
# CPU heads are durable per variant; held-out inference is durable per 16-tile shard.
# ============================================================
cv_started = time.perf_counter()
for fold_i, heldout in enumerate(TRAIN_CONTIGS, 1):
    training_contigs = [c for c in TRAIN_CONTIGS if c != heldout]
    print("\n" + "=" * 88)
    print(f"FOLD {fold_i}/9 — held out {heldout}")
    print("=" * 88)

    paths = fold_eval_paths(fold_i, heldout)
    if paths["complete"].is_file():
        finalize_fold_from_shards(fold_i, heldout, paths)
        print(f"Fold {fold_i} already complete; elapsed={(time.perf_counter()-cv_started)/60:.1f} min")
        continue

    head_A, mean_A, std_A, prior_A, hist_A = ensure_cpu_fold_model(
        fold_i, heldout, training_contigs, "A"
    )
    head_B, mean_B, std_B, prior_B, hist_B = ensure_cpu_fold_model(
        fold_i, heldout, training_contigs, "B"
    )
    model_path_A, _ = fold_model_paths(fold_i, heldout, "A")
    model_path_B, _ = fold_model_paths(fold_i, heldout, "B")

    paths = evaluate_fold_to_atomic_shards(
        fold_i, heldout,
        head_A, mean_A, std_A, prior_A,
        head_B, mean_B, std_B, prior_B,
        model_path_A, model_path_B,
    )
    finalize_fold_from_shards(fold_i, heldout, paths)

    del head_A, head_B
    gc.collect()
    print(f"Fold {fold_i} complete; CV elapsed={(time.perf_counter()-cv_started)/60:.1f} min")

print("\nAll 9 restart-safe CV folds are durable.")



FOLD 1/9 — held out NC_064035.1
[NC_064035.1][A] CPU model already durable — reuse
[NC_064035.1][B] CPU model already durable — reuse
[NC_064035.1] restart-safe CV inference resumes at tile 0/18,918; durable_shards=0


In [ ]:
# ============================================================
# ASSEMBLE POOLED OOF METRICS FROM DURABLE FOLD RESULTS
# This cell does no WIDTH512 inference and is safe to rerun.
# ============================================================
all_fold_metrics = []
all_region_diag = []
all_history = []
pooled_total_hist = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}
pooled_pos_hist = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}
pooled_lowtp_bins = {(v, a): [] for v in ("A", "B") for a in ALPHAS}

for fold_i, heldout in enumerate(TRAIN_CONTIGS, 1):
    paths = fold_eval_paths(fold_i, heldout)
    assert paths["complete"].is_file(), (fold_i, heldout)
    complete = json.loads(paths["complete"].read_text(encoding="utf-8"))
    for name in ("metrics", "regional", "hist"):
        p = paths[name]
        assert sha256_file(p) == complete["outputs_sha256"][p.name]
    all_fold_metrics.append(pd.read_csv(paths["metrics"]))
    all_region_diag.append(pd.read_csv(paths["regional"]))
    for variant in ("A", "B"):
        _, history_path = fold_model_paths(fold_i, heldout, variant)
        all_history.append(pd.read_csv(history_path))
    with np.load(paths["hist"], allow_pickle=False) as h:
        for variant in ("A", "B"):
            for alpha in ALPHAS:
                tag = alpha_tag(alpha)
                pooled_total_hist[(variant, alpha)] += h[f"total_{variant}_{tag}"].astype(np.int64, copy=False)
                pooled_pos_hist[(variant, alpha)] += h[f"pos_{variant}_{tag}"].astype(np.int64, copy=False)
                pooled_lowtp_bins[(variant, alpha)].extend(h[f"lowtp_{variant}_{tag}"].astype(np.int64).tolist())

cv_fold_df = pd.concat(all_fold_metrics, ignore_index=True)
cv_region_diag_df = pd.concat(all_region_diag, ignore_index=True)
training_history_df = pd.concat(all_history, ignore_index=True)
atomic_csv(training_history_df, RUN_DIR / "head_training_history_cpu_v3.csv")
atomic_csv(cv_fold_df, OUTPUTS["cv_fold_ap"])
atomic_csv(cv_region_diag_df, OUTPUTS["cv_region_diag"])

# alpha=0 must exactly reproduce the same baseline ranking for A and B in every held-out contig.
for contig in TRAIN_CONTIGS:
    a0 = cv_fold_df[(cv_fold_df.heldout_contig == contig) & (cv_fold_df.variant == "A") & (np.isclose(cv_fold_df.alpha, 0.0))].iloc[0].ap5
    b0 = cv_fold_df[(cv_fold_df.heldout_contig == contig) & (cv_fold_df.variant == "B") & (np.isclose(cv_fold_df.alpha, 0.0))].iloc[0].ap5
    assert a0 == b0, (contig, a0, b0)

pooled_rows = []
for variant in ("A", "B"):
    for alpha in ALPHAS:
        d = cv_fold_df[(cv_fold_df.variant == variant) & np.isclose(cv_fold_df.alpha, alpha)]
        pooled_rows.append({
            "variant": variant,
            "alpha": alpha,
            "pooled_oof_ap5": ap_from_histograms(pooled_pos_hist[(variant, alpha)], pooled_total_hist[(variant, alpha)]),
            "position_n": int(pooled_total_hist[(variant, alpha)].sum()),
            "positive_n": int(pooled_pos_hist[(variant, alpha)].sum()),
            "mean_fold_ap5": float(d.ap5.mean()),
            "min_fold_ap5": float(d.ap5.min()),
            "max_fold_ap5": float(d.ap5.max()),
        })
cv_pooled_df = pd.DataFrame(pooled_rows)
atomic_csv(cv_pooled_df, OUTPUTS["cv_pooled_ap"])

cv_budget_rows = []
for variant in ("A", "B"):
    for alpha in ALPHAS:
        total_hist = pooled_total_hist[(variant, alpha)]
        pos_hist = pooled_pos_hist[(variant, alpha)]
        n_pos = int(pos_hist.sum())
        positive_bins = np.repeat(np.arange(SUBMISSION_LEVELS, dtype=np.int64), pos_hist)
        low_bins = np.asarray(pooled_lowtp_bins[(variant, alpha)], dtype=np.int64)
        for multiplier in BUDGET_MULTIPLIERS:
            budget = multiplier * n_pos
            cv_budget_rows.append({
                "variant": variant,
                "alpha": alpha,
                "budget_multiplier": multiplier,
                "budget_positions": int(budget),
                "positive_position_recall": tie_aware_recall_at_budget(total_hist, positive_bins, budget),
                "fixed_train_lowtp_recall": tie_aware_recall_at_budget(total_hist, low_bins, budget),
            })
cv_budget_df = pd.DataFrame(cv_budget_rows)
atomic_csv(cv_budget_df, OUTPUTS["cv_budget"])

print("Pooled OOF AP5:")
display(cv_pooled_df.sort_values(["pooled_oof_ap5", "alpha"], ascending=[False, True]))


In [ ]:
# ============================================================
# FREEZE ONE CONFIGURATION FROM TRAIN-CONTIG CV — VALIDATION STILL UNTOUCHED
# ============================================================
selection_table = cv_pooled_df.sort_values(
    ["pooled_oof_ap5", "alpha", "variant"],
    ascending=[False, True, True],
    kind="mergesort",
).reset_index(drop=True)
selected = selection_table.iloc[0]
SELECTED_VARIANT = str(selected["variant"])
SELECTED_ALPHA = float(selected["alpha"])
SELECTED_CV_AP5 = float(selected["pooled_oof_ap5"])

best_A = cv_pooled_df[cv_pooled_df.variant == "A"].sort_values(["pooled_oof_ap5", "alpha"], ascending=[False, True]).iloc[0]
best_B = cv_pooled_df[cv_pooled_df.variant == "B"].sort_values(["pooled_oof_ap5", "alpha"], ascending=[False, True]).iloc[0]

selected_payload = {
    "selected_variant": SELECTED_VARIANT,
    "selected_alpha": SELECTED_ALPHA,
    "selected_pooled_train_cv_ap5": SELECTED_CV_AP5,
    "best_A": {"alpha": float(best_A.alpha), "pooled_oof_ap5": float(best_A.pooled_oof_ap5)},
    "best_B": {"alpha": float(best_B.alpha), "pooled_oof_ap5": float(best_B.pooled_oof_ap5)},
    "B_minus_A_best_ap5": float(best_B.pooled_oof_ap5 - best_A.pooled_oof_ap5),
    "selection_rule": "max pooled train-contig OOF AP5; then smaller alpha; then A lexical tie-break",
    "cv_runtime_amendment": "002",
    "regional_heads_cpu": True,
    "backbone_directml": True,
    "exp046b_validation_inference_or_labels_used_for_selection": False,
}
atomic_json(OUTPUTS["selected_config"], selected_payload)

fig, ax = plt.subplots(figsize=(8, 5))
for variant in ("A", "B"):
    d = cv_pooled_df[cv_pooled_df.variant == variant].sort_values("alpha")
    ax.plot(d.alpha, d.pooled_oof_ap5, marker="o", label=f"Variant {variant}")
ax.set_xlabel("alpha")
ax.set_ylabel("pooled train-contig OOF nucleotide AP5")
ax.set_title("EXP046-B train-contig CV: regional correction strength")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUTS["cv_plot"], dpi=160)
plt.close(fig)

print("CONFIG FROZEN BEFORE VALIDATION")
print(json.dumps(selected_payload, indent=2))


## Raw-score audit

This is post-selection only. Configuration selection above uses the preregistered five-decimal natural nucleotide AP5. The raw-score audit checks whether the selected correction changes ranking beyond rounded-score tie behavior.


In [ ]:
# ============================================================
# POST-SELECTION RAW-SCORE AUDIT — PER HELD-OUT TRAIN CONTIG
# Does not affect selection. Streams durable shards; no new WIDTH512 inference.
# ============================================================
raw_rows = []
for fold_i, contig in enumerate(TRAIN_CONTIGS, 1):
    paths = fold_eval_paths(fold_i, contig)
    model_path_A, _ = fold_model_paths(fold_i, contig, "A")
    model_path_B, _ = fold_model_paths(fold_i, contig, "B")
    shards, frontier = verify_existing_shards(
        paths,
        contig,
        len(train_tiles[train_tiles["contig"].astype(str) == contig]),
        sha256_file(model_path_A),
        sha256_file(model_path_B),
    )
    assert frontier == len(train_tiles[train_tiles["contig"].astype(str) == contig])
    baseline_parts, selected_parts, label_parts = [], [], []
    try:
        for p in shards:
            with np.load(p, allow_pickle=False) as d:
                base = d["base_logit"].astype(np.float32, copy=False)
                valid = u32_to_bool_rows(d["valid_mask"])
                positive = u32_to_bool_rows(d["positive_mask"])
                g = d[f"g_{SELECTED_VARIANT}"].astype(np.float32, copy=False)
                baseline_parts.append(base[valid].copy())
                selected_parts.append((base + np.float32(SELECTED_ALPHA) * g[:, None])[valid].astype(np.float32, copy=True))
                label_parts.append(positive[valid].copy())
        baseline_scores = np.concatenate(baseline_parts)
        selected_scores = np.concatenate(selected_parts)
        labels = np.concatenate(label_parts)
        baseline_raw = exact_ap_from_scores(baseline_scores, labels)
        selected_raw = exact_ap_from_scores(selected_scores, labels)
        raw_rows.append({
            "segment": contig,
            "baseline_raw_ap": baseline_raw,
            "selected_raw_ap": selected_raw,
            "delta_raw_ap": selected_raw - baseline_raw,
        })
        del baseline_scores, selected_scores, labels
    except MemoryError:
        warnings.warn(f"Raw AP audit skipped for {contig} after MemoryError; AP5 CV result remains authoritative")
        raw_rows.append({
            "segment": contig,
            "baseline_raw_ap": float("nan"),
            "selected_raw_ap": float("nan"),
            "delta_raw_ap": float("nan"),
        })
    finally:
        del baseline_parts, selected_parts, label_parts
        gc.collect()

raw_df = pd.DataFrame(raw_rows)
atomic_csv(raw_df, OUTPUTS["cv_raw_audit"])
display(raw_df)


In [ ]:
# ============================================================
# CV RECOVERY COMPLETE — STOP BEFORE VALIDATION
# ============================================================
assert OUTPUTS["selected_config"].is_file()
assert not (RUN_DIR / "validation_nucleotide_metrics.csv").exists()
assert not (RUN_DIR / "EXP046B_RESULT.json").exists()

summary = {
    "experiment_id": EXPERIMENT_ID,
    "run": "run_001",
    "recovery_version": "V3_RESTART_SAFE_CV",
    "amendments": ["001", "002"],
    "selected_from_train_cv": selected_payload,
    "folds_complete": len(TRAIN_CONTIGS),
    "cv_shard_tiles": CV_SHARD_TILES,
    "head_device": "cpu",
    "backbone_device": "DirectML",
    "validation_used": False,
    "next_step": "inspect CV result; then run one separately guarded final train + validation stage only if proceeding",
}
CV_COMPLETE_PATH = RUN_DIR / "EXP046B_CV_V3_COMPLETE.json"
atomic_json(CV_COMPLETE_PATH, summary)
print("=" * 88)
print("EXP046-B V3 TRAIN-CONTIG CV COMPLETE")
print("=" * 88)
print(f"Selected: variant={SELECTED_VARIANT}, alpha={SELECTED_ALPHA}")
print(f"Pooled train-contig OOF AP5: {SELECTED_CV_AP5:.8f}")
print("Validation used:", False)
print("CV completion marker:", CV_COMPLETE_PATH)
print("STOP HERE. Do not run validation from the old V2 notebook.")
